# 05 · Validation plan — activity assay, DSF thermostability, controls, solubility redesign

**Standard slot:** *validation plan.* **For Project 19 this means:** turn the catalytic-geometry-
filtered, **thermostability-ranked** set into a costed **activity-assay plan** (pNP-ester colorimetric
→ PET-film/HPLC) + a **DSF thermostability** plan with the right controls (incl. a catalytic-Ser→Ala
dead mutant) and a **surface-residue redesign for solubility** stretch (D4/D5).

This is the deliverable that states, plainly: **in-silico geometry + an MD proxy are hypotheses; the
activity assay and DSF test them.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the thermostability-ranked, geometry-passing set, capped at **< 96** so they
fit a single screening plate with controls. **Balance the thermostability ↔ activity trade-off** —
don't pick only the most rigid designs — and keep diversity across scaffolds/tracks.

In [ ]:
import pandas as pd
# Prefer the thermostability-ranked, geometry-passing set from nb04; fall back gracefully.
for src in ("results/thermostability_ranked.csv", "results/ranked.csv", "results/campaign.csv"):
    try:
        ranked = pd.read_csv(src); used = src; break
    except FileNotFoundError:
        continue
print("using:", used)

ok = ranked[ranked["catalytic_geom_rmsd"] <= 0.5] if "catalytic_geom_rmsd" in ranked else ranked
if "thermostability_rank_score" in ok:
    ok = ok.sort_values("thermostability_rank_score", ascending=False)
selected = ok.head(88).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds/tracks; include a few high-activity-leaning (less rigid) designs to")
print("probe the thermostability<->activity trade-off; record why each was chosen in your report.")

## 2 · The activity assay (pNP-ester fast screen → PET-film/HPLC true substrate)
- **Express + purify:** *E. coli* BL21(DE3) (or SHuffle for disulfide-containing cutinase-like folds),
  16–18 °C overnight; His-tag → IMAC → SEC polish.
- **Fast screen — pNP-ester colorimetric:** p-nitrophenyl acetate/butyrate; follow released
  **p-nitrophenolate at ~405–410 nm** in a plate reader; take initial rates. (A soluble-ester proxy
  for throughput — *not* PET itself.)
- **True substrate — PET-film / amorphous-PET digestion + HPLC:** incubate hits with amorphous PET
  film/powder; quantify released **MHET / TPA** by HPLC over time and **temperature** (30 / 50 / 65 °C)
  — the real plastic-degradation readout, and where thermostability pays off.
- **Readout note:** subtract buffer-only background; for PET, surface area / crystallinity strongly
  affect rates — standardise the substrate.

## 3 · DSF thermostability + the mandatory controls
The whole project is optimised for thermostability, so **measure it**: differential scanning
fluorimetry (DSF / thermal shift) → report **Tm**, compared to a natural reference. Then the controls
every plate needs.

In [ ]:
controls = {
    "POSITIVE — natural/reference PET hydrolase": "a verified IsPETase / LCC / thermostable cutinase; "
        "confirms the assay works and benchmarks activity + Tm",
    "NEGATIVE — catalytic Ser -> Ala 'dead' mutant": "SAME design, catalytic Ser mutated to Ala; "
        "cleanest negative — loss of activity pins catalysis to the nucleophile (His/Asp inactive without it)",
    "NEGATIVE — heat-killed enzyme": "boiled aliquot; rules out non-protein / contaminant ester hydrolysis",
    "NEGATIVE — empty-vector lysate": "no insert; rules out host-background esterase activity",
    "BLANK — buffer + substrate only": "the uncatalysed background ester hydrolysis to subtract",
}
print("MANDATORY controls (every plate):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")
print("\nDSF: run thermal melts on each design + the natural reference; report Tm (deg C). This is the")
print("measurement the campaign's thermostability ranking is trying to predict — close the loop here.")

## 4 · A costed, plate-based screen (template — fill real prices)
One 96-well plate holds the < 96 designs + the controls above. Cost the gene synthesis, expression,
the pNP-ester + PET-film/HPLC assays, and DSF at your institution's rates; the cell prints a template
to fill in your report.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis", "1 plate", "fill"),
    ("IMAC purification (plate format)", "1 plate", "fill"),
    ("pNP-ester substrate (pNP-acetate/butyrate)", "stock", "fill"),
    ("Amorphous PET film/powder + HPLC consumables (MHET/TPA)", "per assay", "fill"),
    ("DSF / thermal-shift reagents + instrument time", "per plate", "fill"),
    ("Plate-reader time (pNP kinetics)", "per plate", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:54s} {scale:12s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+pNP screen 1 wk ->")
print("                    DSF + PET-film/HPLC on hits 1-2 wk.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here —")
print("industrial/green-chemistry — but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 5 · Surface-residue redesign for solubility `[stretch]`
A thermostable hit that won't express solubly is useless. If a top design is poorly soluble, redesign
**only the surface** residues (LigandMPNN / ProteinMPNN with the **triad + core fixed**) to improve
solubility/expression — then **re-check the catalytic geometry and the thermostability proxy**, since
surface changes can shift both. (Optionally, sketch a directed-evolution loop for activity using the
pNP screen as the readout — the honest history is that designs often need it.)

In [ ]:
print("Surface-redesign-for-solubility loop (stretch):")
print("  pick a thermostable hit with poor predicted solubility ->")
print("  LigandMPNN/ProteinMPNN redesign SURFACE only (triad + oxyanion hole + core FIXED) ->")
print("  re-predict (AF2) -> re-check catalytic_geometry_rmsd AND the thermostability proxy ->")
print("  keep variants that improve solubility WITHOUT breaking geometry/stability.")
print("\nReminder for the thesis: report the hit rate honestly, and that a thermostable, well-folded")
print("design can still be catalytically inactive (geometry != activity; MD proxy != Tm). The assay decides.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, catalytic-geometry-passing, thermostability-ranked designs.
- [ ] Activity-assay plan: pNP-ester fast screen + PET-film/HPLC true substrate, initial rates / kinetics.
- [ ] **DSF thermostability** plan (report Tm vs a natural reference) — close the loop on the ranking.
- [ ] **All** controls (catalytic-Ser→Ala dead mutant, heat-killed, empty vector, blank), costed + timed.
- [ ] Surface-redesign-for-solubility (and optional directed-evolution) plan for hits `[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "geometry ≠ activity" + "MD proxy ≠ Tm" stated plainly.

You're done — this project followed the Project 18 theozyme→scaffold→sequence→geometry template, with ester hydrolysis + a **thermostability** emphasis.